In [42]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import OneHotEncoder, LabelEncoder, StandardScaler, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer

from pathlib import Path

In [43]:
# load the dataset
path_to_dataset = Path.cwd().parent / 'dataset'    # make sure the folder that
# contains the dataset is named "dataset"

train_df = pd.read_csv(path_to_dataset / 'train.csv')

In [44]:
# work on this dataset use the other as a reference to the original one
preprocessed_df = train_df.copy()

In [45]:
# TASK 1:
# start by dropping the unnecessary columns here
# then print insights about the dataset
# and the number of categorical and numerical features
# and the cardinality of the remaining categorical features
# and the number of missing values in each feature

In [46]:
cols_to_drop= ['id','recorded_by','num_private', 'scheme_name','wpt_name', 'subvillage',
               'ward', 'date_recorded', 'region_code', 'district_code', 'quantity_group',
               'payment_type', 'quality_group', 'source_type', 'waterpoint_type_group',
               'extraction_type_group', 'management_group', 'public_meeting', 'scheme_management']
preprocessed_df = preprocessed_df.drop(columns= cols_to_drop)

In [47]:
print(preprocessed_df.shape)

(59400, 22)


#### Deleted 19 features aka cols so the cols count is 22 instead of 41

In [48]:
print(preprocessed_df.dtypes)

amount_tsh               float64
funder                       str
gps_height                 int64
installer                    str
longitude                float64
latitude                 float64
basin                        str
region                       str
lga                          str
population                 int64
permit                    object
construction_year          int64
extraction_type              str
extraction_type_class        str
management                   str
payment                      str
water_quality                str
quantity                     str
source                       str
source_class                 str
waterpoint_type              str
status_group                 str
dtype: object


### Now we have 16 categorical features and 6 numerical features which are:

In [49]:
print(preprocessed_df.columns)

Index(['amount_tsh', 'funder', 'gps_height', 'installer', 'longitude',
       'latitude', 'basin', 'region', 'lga', 'population', 'permit',
       'construction_year', 'extraction_type', 'extraction_type_class',
       'management', 'payment', 'water_quality', 'quantity', 'source',
       'source_class', 'waterpoint_type', 'status_group'],
      dtype='str')


### Checking missing values per feature:

In [50]:
print(preprocessed_df.isnull().sum()) 

amount_tsh                  0
funder                   3637
gps_height                  0
installer                3655
longitude                   0
latitude                    0
basin                       0
region                      0
lga                         0
population                  0
permit                   3056
construction_year           0
extraction_type             0
extraction_type_class       0
management                  0
payment                     0
water_quality               0
quantity                    0
source                      0
source_class                0
waterpoint_type             0
status_group                0
dtype: int64


### Cardinality of categorical features:

In [51]:
print(preprocessed_df.select_dtypes(include='object').nunique())

funder                   1896
installer                2145
basin                       9
region                     21
lga                       125
permit                      2
extraction_type            18
extraction_type_class       7
management                 12
payment                     7
water_quality               8
quantity                    5
source                     10
source_class                3
waterpoint_type             7
status_group                3
dtype: int64


C:\Users\abdul\AppData\Local\Temp\ipykernel_7272\1154399320.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  print(preprocessed_df.select_dtypes(include='object').nunique())


### Checking duplicate rows: 

In [52]:
print(preprocessed_df.duplicated().sum()) 

1106


### Statistical summary for numerical and categorical data:

In [53]:
print(preprocessed_df.describe(include='all'))

           amount_tsh                  funder    gps_height installer  \
count    59400.000000                   55763  59400.000000     55745   
unique            NaN                    1896           NaN      2145   
top               NaN  Government Of Tanzania           NaN       DWE   
freq              NaN                    9084           NaN     17402   
mean       317.650385                     NaN    668.297239       NaN   
std       2997.574558                     NaN    693.116350       NaN   
min          0.000000                     NaN    -90.000000       NaN   
25%          0.000000                     NaN      0.000000       NaN   
50%          0.000000                     NaN    369.000000       NaN   
75%         20.000000                     NaN   1319.250000       NaN   
max     350000.000000                     NaN   2770.000000       NaN   

           longitude      latitude          basin  region     lga  \
count   59400.000000  5.940000e+04          59400   59

In [54]:
# TASK 2:

# handle the missing values in the dataset here 
# bonus task: try to find a way to handle the missing values in the most efficient way possible
# if u can see a pattern in the missing values, try to use it to fill the missing values using 
# MAR, MCAR, MNAR techniques

### Start: Replacing 0 With NaN (Following merged_conc)

In [55]:
preprocessed_df.loc[preprocessed_df['amount_tsh'] == 0, 'amount_tsh'] = np.nan
preprocessed_df.loc[preprocessed_df['construction_year'] == 0, 'construction_year'] = np.nan
preprocessed_df.loc[preprocessed_df['population'] == 0, 'population'] = np.nan
preprocessed_df.loc[preprocessed_df['longitude'] == 0, 'longitude'] = np.nan
preprocessed_df.loc[preprocessed_df['gps_height'] <= 0, 'gps_height'] = np.nan

### 1) Fill With Unknown (MCAR)

In [56]:
preprocessed_df['funder'] = preprocessed_df['funder'].fillna('Unknown')
preprocessed_df['installer'] = preprocessed_df['installer'].fillna('Unknown')

### 2) Fill With Mode (MNAR)

In [57]:
preprocessed_df['permit'] = preprocessed_df['permit'].fillna(preprocessed_df['permit'].mode()[0])

### 3) Fill With Median

In [58]:
#First, find relations between missing values and the other columns (the ones that don't have missing values)

cols_to_fix = ['amount_tsh', 'population', 'construction_year', 'gps_height', 'longitude']
complete_cols = preprocessed_df.columns[preprocessed_df.isnull().sum() == 0].tolist()
complete_cols = [c for c in complete_cols if c not in ['funder', 'installer', 'status_group']]

for col in cols_to_fix:
    print(f"\n{col}:")
    
    for other in complete_cols:
        missing_by_group = preprocessed_df.groupby(other)[col].apply(lambda x: x.isna().mean() * 100)
        if len(missing_by_group) > 1:
            diff = missing_by_group.max() - missing_by_group.min()
            if diff > 30:
                print(f"  {other}: range = {missing_by_group.min():.1f}% - {missing_by_group.max():.1f}%")


amount_tsh:
  latitude: range = 0.0% - 100.0%
  basin: range = 49.0% - 95.8%
  region: range = 33.5% - 100.0%
  lga: range = 1.8% - 100.0%
  extraction_type: range = 39.8% - 100.0%
  extraction_type_class: range = 41.6% - 91.7%
  management: range = 30.3% - 94.6%
  payment: range = 15.2% - 99.8%
  water_quality: range = 68.4% - 100.0%
  source: range = 48.1% - 95.3%
  waterpoint_type: range = 60.6% - 100.0%

population:
  latitude: range = 0.0% - 100.0%
  basin: range = 0.0% - 75.3%
  region: range = 0.0% - 100.0%
  lga: range = 0.0% - 100.0%
  extraction_type: range = 14.2% - 98.9%
  extraction_type_class: range = 14.2% - 52.5%
  management: range = 0.0% - 65.1%
  payment: range = 16.8% - 62.7%
  water_quality: range = 16.5% - 82.5%
  source: range = 13.2% - 55.6%
  waterpoint_type: range = 14.3% - 53.1%

construction_year:
  latitude: range = 0.0% - 100.0%
  basin: range = 1.7% - 75.6%
  region: range = 0.1% - 100.0%
  lga: range = 0.0% - 100.0%
  extraction_type: range = 13.5% - 98

#### Conclusion: Ignoring `latitude` and `lga` as their numbers are too consistently strange.
####             This leaves the columns with the strongest variation (showing a stronger dependance on each group) being the following:
####             amount_tsh -> region, population -> region, gps_height -> region, longitude -> extraction_type

In [59]:
#Now actually start replacing with the median of each group following MAR 
#except for construction_year, as per merged_conc, which would be MCAR

preprocessed_df['construction_year'] = preprocessed_df['construction_year'].fillna(1986)
preprocessed_df['amount_tsh'] = preprocessed_df.groupby('region')['amount_tsh'].transform(lambda x: x.fillna(x.median()))
preprocessed_df['population'] = preprocessed_df.groupby('region')['population'].transform(lambda x: x.fillna(x.median()))
preprocessed_df['gps_height'] = preprocessed_df.groupby('region')['gps_height'].transform(lambda x: x.fillna(x.median()))
preprocessed_df['longitude'] = preprocessed_df.groupby('extraction_type')['longitude'].transform(lambda x: x.fillna(x.median()))

cols_to_fix = ['amount_tsh', 'population', 'gps_height', 'longitude']
preprocessed_df[cols_to_fix] = preprocessed_df[cols_to_fix].fillna(preprocessed_df[cols_to_fix].median()) #For safety, fill with regular median

In [60]:
print(preprocessed_df.isnull().sum())

amount_tsh               0
funder                   0
gps_height               0
installer                0
longitude                0
latitude                 0
basin                    0
region                   0
lga                      0
population               0
permit                   0
construction_year        0
extraction_type          0
extraction_type_class    0
management               0
payment                  0
water_quality            0
quantity                 0
source                   0
source_class             0
waterpoint_type          0
status_group             0
dtype: int64


In [61]:
# TASK 3:

# check each categorical feature to see if 
# they are nominal or ordinal

In [62]:
# Before I start I need to handle some speacial cases in the categorical features 3lshan ana 5adt baly mn shwaya habal zy:
# funder and installer have 1896 and 2145 unique values
# keeping all of them would create too many columns after encoding
# solution: keep only top 10 most frequent, group the rest as 'Other'
# But for lga has 125 unique values, same treatment but keep top 20 (why? because it's a veryimportant feature as it determines the location of the pump and thus the water availability and quality)
# also clean typos in installer as some of them belong to the same type Ex: (DANIDA / DANID / Danida -> same thing)

def keep_top_n(df, col, n):
    top = df[col].value_counts().nlargest(n).index
    df[col] = df[col].apply(lambda x: x if x in top else 'Other')
    return df

preprocessed_df = keep_top_n(preprocessed_df, 'funder', n=10)
preprocessed_df = keep_top_n(preprocessed_df, 'installer', n=10)
preprocessed_df = keep_top_n(preprocessed_df, 'lga', n=20)

# clean typos in installer (DANIDA / DANID / Danida -> same thing)
preprocessed_df['installer'] = preprocessed_df['installer'].str.strip().str.lower().str.title()
preprocessed_df['funder'] = preprocessed_df['funder'].str.strip().str.lower().str.title()

print("funder unique after grouping:", preprocessed_df['funder'].nunique())
print("installer unique after grouping:", preprocessed_df['installer'].nunique())
print("lga unique after grouping:", preprocessed_df['lga'].nunique())

funder unique after grouping: 11
installer unique after grouping: 11
lga unique after grouping: 21


In [63]:
# CHECK NOMINAL VS ORDINAL

# ALL remaining categorical features in this dataset are NOMINAL:
# - basin, region, extraction_type, extraction_type_class,
#   management, payment, water_quality, quantity, source,
#   source_class, waterpoint_type, permit, funder, installer, lga

# None of them have a natural ranking order so we use OneHotEncoding for all of them

# Note about quantity:
# quantity may looks shwaya ordinal (enough > insufficient > seasonal > dry)
# bas howa la2 as 'enough' vs 'insufficient' is not a scale,
# it's a category describing the water availability state
# so OneHot is still correct here

In [64]:
# apply the appropriate encoding technique for each feature
# [ONE-HOT ENCODE ALL CATEGORICAL FEATURES]

# bagyb kol categorical columns except the target 'status_group'
cat_cols = preprocessed_df.select_dtypes(include='object').columns.tolist()
cat_cols = [c for c in cat_cols if c != 'status_group']

print("Categorical columns to encode:", cat_cols)
print("Total:", len(cat_cols))

# drop_first=False → keep all categories 3lshan a7na hngrab f kaza model f this is important for [tree models] msln
preprocessed_df = pd.get_dummies(preprocessed_df, columns=cat_cols, drop_first=False)

# convert boolean columns created by get_dummies to int (0/1) instead of (True/False) 3lshan el models bt7eb kda
bool_cols = preprocessed_df.select_dtypes(include='bool').columns
preprocessed_df[bool_cols] = preprocessed_df[bool_cols].astype(int)

print("\nShape after encoding:", preprocessed_df.shape)

Categorical columns to encode: ['funder', 'installer', 'basin', 'region', 'lga', 'permit', 'extraction_type', 'extraction_type_class', 'management', 'payment', 'water_quality', 'quantity', 'source', 'source_class', 'waterpoint_type']
Total: 15

Shape after encoding: (59400, 159)


C:\Users\abdul\AppData\Local\Temp\ipykernel_7272\2914740849.py:5: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = preprocessed_df.select_dtypes(include='object').columns.tolist()


In [65]:
# check for numerical features and apply the appropriate scaling technique for each feature

# here I have to handle the highly skewed (el tail el taweeel) features before scaling so I will use:
# LOG1P TO TRANSFORM SKEWED NUMERICAL FEATURES 

# amount_tsh: skewness 57.8 → extremely right skewed
# population: skewness 12.7 → shwaya right skewed

# log1p = log(x + 1), the +1 handles zeros safely as log(0) is undefined

preprocessed_df['amount_tsh'] = np.log1p(preprocessed_df['amount_tsh'])
preprocessed_df['population'] = np.log1p(preprocessed_df['population'])

print("amount_tsh skewness after log1p:", preprocessed_df['amount_tsh'].skew().round(3))
print("population skewness after log1p:", preprocessed_df['population'].skew().round(3))


amount_tsh skewness after log1p: -0.102
population skewness after log1p: -1.361


In [66]:
# SCALE NUMERICAL FEATURES
# We use RobustScaler instead of StandardScaler because: (hktb el reasons cuz a5yraaan basmaga el manhag b2et leha lazma)
# - StandardScaler is sensitive to outliers (uses mean and std)
# - RobustScaler uses median and IQR so outliers don't affect the scaling
# - our numerical features still have outliers even after log transform (but they are less extreme now so RobustScaler can handle them better)

# columns to scale: all remaining numeric except target-related ones

preprocessed_df['year_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.year
preprocessed_df['month_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.month

num_cols = ['amount_tsh', 'gps_height', 'longitude', 'latitude',
            'population', 'construction_year', 'year_recorded', 'month_recorded']

scaler = RobustScaler()
preprocessed_df[num_cols] = scaler.fit_transform(preprocessed_df[num_cols])

print("\nNumerical features after scaling:")
print(preprocessed_df[num_cols].describe().round(3))


Numerical features after scaling:
       amount_tsh  gps_height  longitude   latitude  population  \
count   59400.000   59400.000  59400.000  59400.000   59400.000   
mean       -0.135      -0.204      0.044     -0.131      -0.312   
std         0.618       0.980      0.671      0.565       1.084   
min        -2.339      -2.311     -1.392     -1.271      -2.882   
25%        -0.697      -0.688     -0.418     -0.675      -0.568   
50%         0.000       0.000      0.000     -0.000       0.000   
75%         0.303       0.312      0.582      0.325       0.432   
max         3.169       3.071      1.401      0.963       3.140   

       construction_year  year_recorded  month_recorded  
count          59400.000      59400.000       59400.000  
mean               0.391         -0.039           0.275  
std                0.628          0.479           0.606  
min               -1.444         -5.000          -0.400  
25%                0.000         -0.500          -0.200  
50%          

C:\Users\abdul\AppData\Local\Temp\ipykernel_7272\149717160.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  preprocessed_df['year_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.year
C:\Users\abdul\AppData\Local\Temp\ipykernel_7272\149717160.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  preprocessed_df['month_recorded'] = pd.to_datetime(train_df['date_recorded']).dt.month


In [67]:
# after scaling check if there are any outliers in the dataset and handle them using the appropriate technique

# check if extreme outliers still exist using IQR

print("\nOutlier check after scaling (IQR method):")
for col in num_cols:
    Q1 = preprocessed_df[col].quantile(0.25)
    Q3 = preprocessed_df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    n_out = ((preprocessed_df[col] < lower) | (preprocessed_df[col] > upper)).sum()
    print(f"  {col}: {n_out} outliers ({n_out/len(preprocessed_df)*100:.1f}%)")


Outlier check after scaling (IQR method):
  amount_tsh: 109 outliers (0.2%)
  gps_height: 2598 outliers (4.4%)
  longitude: 0 outliers (0.0%)
  latitude: 0 outliers (0.0%)
  population: 7252 outliers (12.2%)
  construction_year: 0 outliers (0.0%)
  year_recorded: 31 outliers (0.1%)
  month_recorded: 0 outliers (0.0%)


In [68]:
# Final Shape after Task 3:
print("\nFinal shape after Task 3:", preprocessed_df.shape)
print("Ready for Task 4")


Final shape after Task 3: (59400, 161)
Ready for Task 4


In [69]:
# TASK 4:

# after handling the data
# check the correlation between the features and the target variable
# look at their variance too and try to only without dropping anything
# check if there are any features that have low variance which can indicate
# that they are not useful for the model and can be dropped

In [70]:
# check for multicollinearity between the features again (aka if the features are highly correlated with each other)
# and if they are check for variance too
# dont do anything after only say ur conclusions and what u think we should drop

In [71]:
# create the extra features that we will be using in the model (if u think there are any)

In [72]:
# NOTE WE WONT BE USING PCA BECAUSE IT IS EASIER TO EXPLAIN THE MODEL WITHOUT IT
# AND TO KEEP INTERPRETABILITY + THE DATASET IS NOT THAT COMPLEX
# AND NORMAL FEATURE SELECTION TECHNIQUES SHOULD BE ENOUGH